# 06_mock: Concurrent Web Crawler (Reported Repeatedly)

Timebox: **55 minutes**  
Language: **Python (Colab)**

## Scenario
Implement a same-host crawler first in single-thread mode, then in multi-thread mode.

## What to implement
1. `normalize_url`
2. `crawl_single_thread`
3. `crawl_multi_thread`

## Completion criteria (required)
- Fragment normalization (`#...`) and consistent URL handling
- Same-host filtering
- Each URL fetched at most once
- Matching output between single-thread and multi-thread implementations

## Time guidance
- 10 min: define URL normalization and dedupe rules
- 35 min: implement single-thread then multi-thread crawler
- 10 min: run tests and inspect race-condition risks


In [ ]:
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from copy import deepcopy
from threading import Lock
from typing import Any
from urllib.parse import urldefrag, urlparse
import time

WEB_GRAPH = {
    "https://docs.local/start": [
        "https://docs.local/a#intro",
        "https://docs.local/b",
        "https://external.com/ignore",
    ],
    "https://docs.local/a": [
        "https://docs.local/b",
        "https://docs.local/c",
    ],
    "https://docs.local/b": [
        "https://docs.local/c#part",
        "https://docs.local/d",
    ],
    "https://docs.local/c": [
        "https://docs.local/start",
    ],
    "https://docs.local/d": [],
}


class FakeHtmlParser:
    def __init__(self, graph: dict[str, list[str]], delay_seconds: float = 0.0) -> None:
        self._graph = deepcopy(graph)
        self._delay_seconds = delay_seconds
        self._calls: list[str] = []
        self._lock = Lock()

    def getUrls(self, url: str) -> list[str]:
        if self._delay_seconds:
            time.sleep(self._delay_seconds)
        with self._lock:
            self._calls.append(url)
        return deepcopy(self._graph.get(url, []))

    def call_count(self) -> int:
        with self._lock:
            return len(self._calls)


In [ ]:
def normalize_url(url: str) -> str:
    """Remove URL fragments and normalize simple trailing-slash variants."""
    # TODO
    raise NotImplementedError


def crawl_single_thread(start_url: str, parser: Any) -> list[str]:
    """
    Crawl only URLs on the same hostname as start_url.
    Requirements:
    - Fetch each normalized URL at most once.
    - Ignore cross-host links.
    - Return sorted list of visited URLs.
    """
    # TODO
    raise NotImplementedError


def crawl_multi_thread(start_url: str, parser: Any, max_workers: int = 4) -> list[str]:
    """
    Same behavior as single-thread crawler but using ThreadPoolExecutor.
    Keep behavior deterministic via sorted return.
    """
    # TODO
    raise NotImplementedError


## Run Tests
Run this final test cell after implementing all TODO sections.


In [ ]:
def run_exam06_tests() -> None:
    expected = [
        "https://docs.local/a",
        "https://docs.local/b",
        "https://docs.local/c",
        "https://docs.local/d",
        "https://docs.local/start",
    ]

    parser_single = FakeHtmlParser(WEB_GRAPH, delay_seconds=0.0)
    single = crawl_single_thread("https://docs.local/start#home", parser_single)
    assert single == expected
    assert parser_single.call_count() == len(expected)

    parser_multi = FakeHtmlParser(WEB_GRAPH, delay_seconds=0.01)
    multi = crawl_multi_thread("https://docs.local/start#home", parser_multi, max_workers=4)
    assert multi == expected
    assert parser_multi.call_count() == len(expected)

    assert single == multi
    print("06_mock tests passed")


run_exam06_tests()
